# Sales 2025 SKU Real vs Edit Visualization

Notebook ini fokus ke satu tujuan: melihat dampak penyamaan SKU terhadap trend `CF` di `Sales 2025`.

Rule alignment:
- BIG 1625 / 1.625L -> BIG 1L terbaru
- BIG 3100 / 3.1L -> BIG 3L terbaru
- BIG 400ml dengan flavour sama -> SKU 400ml terbaru
- BIG Nipis 350ml -> deskripsi terbaru, SKU tetap sama
- VOLT 200ml -> SKU/deskripsi 24-pack terbaru
- Jika SKU 12-pack diarahkan ke 24-pack, `CF Edit = CF Real / 2`

In [ ]:
# ==========================================
# 1. SALES 2025 SKU REAL VS EDIT VISUALIZATION
# ==========================================
# Memuat library utama dan menyiapkan opsi tampilan dataframe/chart.

from pathlib import Path
import re
import zipfile

import numpy as np
import pandas as pd

try:
    import plotly.express as px
except ModuleNotFoundError:
    px = None

try:
    import matplotlib.pyplot as plt
    from matplotlib.ticker import FuncFormatter
except ModuleNotFoundError:
    plt = None
    FuncFormatter = None

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 220)

## 1. Source File

Notebook ini hanya memakai file dari Google Drive. Ubah path di bawah kalau lokasi file di Drive berbeda.

In [ ]:
# ==========================================
# 2. 1. SOURCE FILE
# ==========================================
# Menentukan lokasi file dashboard di Google Drive dan memastikan file Excel dapat dibaca.

DRIVE_SALES_FILE = Path("/content/drive/MyDrive/Data Sales/Indonesia Sales Dashboard 2026.xlsm")

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ModuleNotFoundError:
    print("Run this notebook in Colab, or mount Google Drive manually before running.")

if not DRIVE_SALES_FILE.exists():
    raise FileNotFoundError(f"File not found: {DRIVE_SALES_FILE}")

with zipfile.ZipFile(DRIVE_SALES_FILE) as zf:
    if "[Content_Types].xml" not in zf.namelist():
        raise ValueError(f"Not a readable Excel workbook: {DRIVE_SALES_FILE}")

SALES_FILE = DRIVE_SALES_FILE
print("Using:", SALES_FILE)

## 2. Load Data

Chart hanya memakai `Sales 2025`. `Sales 2026` dipakai sebagai acuan SKU terbaru.

In [ ]:
# ==========================================
# 3. 2. LOAD DATA
# ==========================================
# Membaca sheet Sales 2025 sebagai data utama dan Sales 2026 sebagai acuan SKU terbaru.

sales25 = pd.read_excel(SALES_FILE, sheet_name="Sales 2025", engine="openpyxl")
sales26 = pd.read_excel(SALES_FILE, sheet_name="Sales 2026", engine="openpyxl")

print("Sales 2025:", sales25.shape)
print("Sales 2026 reference:", sales26.shape)
display(sales25.head(3))

## 3. Clean Real SKU

In [ ]:
# ==========================================
# 4. 3. CLEAN REAL SKU
# ==========================================
# Membersihkan kolom SKU real, format, flavor, dan membuat Group Key untuk proses alignment.

def clean_text(value):
    if pd.isna(value):
        return ""
    value = str(value).strip().upper()
    return re.sub(r"\s+", " ", value)


def clean_code(value):
    if pd.isna(value):
        return ""
    value = str(value).strip()
    return value[:-2] if value.endswith(".0") else value


def group_key(brand, flavor, fmt):
    brand = clean_text(brand)
    flavor = clean_text(flavor)
    if pd.isna(fmt):
        return ""
    fmt = float(fmt)

    if brand == "BIG" and "NIPIS" in flavor and np.isclose(fmt, 0.35):
        return "BIG_NIPIS_350"
    if brand == "BIG" and np.isclose(fmt, 0.4):
        return "BIG_400"
    if brand == "BIG" and (np.isclose(fmt, 1.625) or np.isclose(fmt, 1.0)):
        return "BIG_1L"
    if brand == "BIG" and (np.isclose(fmt, 3.1) or np.isclose(fmt, 3.0)):
        return "BIG_3L"
    if brand == "VOLT" and np.isclose(fmt, 0.2):
        return "VOLT_200_24"
    return ""


actual = pd.DataFrame({
    "Date": pd.to_datetime(sales25["Date"], errors="coerce"),
    "Channel Group": sales25["Channel Group"],
    "Branch": sales25["Branch"],
    "Channel": sales25["Channel"],
    "Cust Code": sales25["Cust Code"].map(clean_code),
    "Customer Name": sales25["Customer Name"],
    "Brand": sales25["Brand"].map(clean_text),
    "Flavor": sales25["Flavor"].map(clean_text),
    "Format": pd.to_numeric(sales25["Format"], errors="coerce"),
    "Box Content": sales25["Box Content"].map(clean_code),
    "CF": pd.to_numeric(sales25["CF"], errors="coerce").fillna(0),
    "Item Code Real": sales25["Item Code (Real)"].map(clean_code),
    "Short Item Description Real": sales25["Short Item Description (Edit).1"].map(clean_text),
})

actual = actual.dropna(subset=["Date"]).copy()
actual["Month"] = actual["Date"].dt.to_period("M").dt.to_timestamp()
actual["Group Key"] = [group_key(b, f, s) for b, f, s in zip(actual["Brand"], actual["Flavor"], actual["Format"])]
actual["SKU Real"] = actual["Item Code Real"] + " | " + actual["Short Item Description Real"]

display(actual.head(10))

## 4. Build Latest SKU Reference

Reference diambil dari `Sales 2026`, lalu dipilih row terbaru per Brand + Flavor + Group Key. Untuk VOLT, reference dibatasi ke `Box Content = 24`.

In [ ]:
# ==========================================
# 5. 4. BUILD LATEST SKU REFERENCE
# ==========================================
# Membangun tabel acuan SKU terbaru dari Sales 2026 berdasarkan Brand, Flavor, dan Group Key.

ref = pd.DataFrame({
    "Reference Date": pd.to_datetime(sales26["fecha_liquidacion"], errors="coerce"),
    "Brand": sales26["desc_marca"].map(clean_text),
    "Flavor": sales26["desc_sabor"].map(clean_text),
    "Format": pd.to_numeric(sales26["desc_formato"], errors="coerce"),
    "Box Content": sales26["cant_contenido"].map(clean_code),
    "CF": pd.to_numeric(sales26["CF"], errors="coerce").fillna(0),
    "Item Code Edit": sales26["cod_articulo"].map(clean_code),
    "Short Item Description Edit": sales26["desc_articulo_corto"].map(clean_text),
})

ref = ref.dropna(subset=["Reference Date"]).copy()
ref["Group Key"] = [group_key(b, f, s) for b, f, s in zip(ref["Brand"], ref["Flavor"], ref["Format"])]
ref = ref[ref["Group Key"] != ""].copy()
ref = ref[(ref["Group Key"] != "VOLT_200_24") | (ref["Box Content"] == "24")]

sku_reference = (
    ref.sort_values(["Brand", "Flavor", "Group Key", "Reference Date", "CF"], ascending=[True, True, True, False, False])
    .drop_duplicates(["Brand", "Flavor", "Group Key"])
    [[
        "Brand", "Flavor", "Group Key", "Reference Date",
        "Item Code Edit", "Short Item Description Edit", "Format", "Box Content",
    ]]
    .rename(columns={"Format": "Reference Format", "Box Content": "Reference Box Content"})
)

display(sku_reference.sort_values(["Brand", "Group Key", "Flavor"]))

## 5. Apply SKU Edit

In [ ]:
# ==========================================
# 1. GABUNGKAN DATA TRANSAKSI & REFERENSI
# ==========================================
# Melakukan LEFT JOIN antara data aktual dan acuan SKU berdasarkan Brand, Flavor, dan Group Key
aligned = actual.merge(sku_reference, on=["Brand", "Flavor", "Group Key"], how="left")

# Syarat acuan valid: Group Key tidak kosong dan ada Item Code Edit dari tabel acuan
has_reference = aligned["Group Key"].ne("") & aligned["Item Code Edit"].notna()

# ==========================================
# 2. UPDATE KODE & DESKRIPSI ITEM
# ==========================================
# Jika punya acuan valid, pakai Item Code Edit terbaru; jika tidak, pakai Item Code Real
aligned["Item Code Edit"] = np.where(has_reference, aligned["Item Code Edit"], aligned["Item Code Real"])

# Jika punya acuan valid, pakai deskripsi terbaru; jika tidak, pakai deskripsi real
aligned["Short Item Description Edit"] = np.where(
    has_reference,
    aligned["Short Item Description Edit"],
    aligned["Short Item Description Real"],
)

# ==========================================
# 3. FORMAT SKU EDIT & DETEKSI PERUBAHAN
# ==========================================
# Menggabungkan kode dan deskripsi menjadi format audit 'KODE | DESKRIPSI'
aligned["SKU Edit"] = aligned["Item Code Edit"].map(clean_code) + " | " + aligned["Short Item Description Edit"].map(clean_text)

# Menandai apakah terjadi perubahan dari SKU Real ke SKU Edit
aligned["Real != Edit"] = aligned["SKU Real"] != aligned["SKU Edit"]

# ==========================================
# 4. PENYESUAIAN CASE FACTOR (CF)
# ==========================================
# Menyimpan nilai CF awal sebelum adjustment
aligned["CF Real"] = aligned["CF"]

# Syarat adjustment: pack lama 12, acuan baru 24, dan SKU/deskripsi berubah
pack_12_to_24 = aligned["Box Content"].eq("12") & aligned["Reference Box Content"].eq("24") & aligned["Real != Edit"]

# Jika 12-pack diarahkan ke 24-pack, CF dibagi 2; selain itu CF tetap sama
aligned["CF Edit"] = np.where(pack_12_to_24, aligned["CF Real"] / 2, aligned["CF Real"])

# Memberi label status adjustment untuk audit
aligned["CF Adjustment"] = np.where(pack_12_to_24, "12-pack to 24-pack: CF / 2", "No CF adjustment")

# ==========================================
# 5. SELEKSI KOLOM & METADATA WAKTU
# ==========================================
# Mengambil kolom final yang dibutuhkan untuk visualisasi dan forecast
sales_2025_visual = aligned[[
    "Date", "Channel Group", "Branch", "Channel", "Cust Code", "Customer Name",
    "Short Item Description Real", "Item Code Real",
    "Short Item Description Edit", "Item Code Edit",
    "Format", "Box Content", "Reference Box Content",
    "CF Real", "CF Edit", "CF Adjustment",
    "Group Key", "SKU Real", "SKU Edit", "Real != Edit",
]].copy()

# Menambahkan identitas sheet sumber dan hierarki tanggal
sales_2025_visual.insert(0, "Source Sheet", "Sales 2025")
sales_2025_visual.insert(1, "Year", sales_2025_visual["Date"].dt.year)
sales_2025_visual.insert(2, "Month No", sales_2025_visual["Date"].dt.month)

# Membuat timestamp bulanan untuk grouping dan chart
sales_2025_visual["Month"] = sales_2025_visual["Date"].dt.to_period("M").dt.to_timestamp()

# ==========================================
# 6. RINGKASAN OUTPUT & TAMPILAN DATA
# ==========================================
# Menampilkan total baris output Sales 2025 setelah alignment
print("Rows:", len(sales_2025_visual))

# Menampilkan total baris yang berubah dari SKU Real ke SKU Edit
print("Rows changed Real -> Edit:", int(sales_2025_visual["Real != Edit"].sum()))

# Menampilkan total baris yang mendapatkan CF adjustment / 2
print("Rows with CF / 2 adjustment:", int((sales_2025_visual["CF Adjustment"] == "12-pack to 24-pack: CF / 2").sum()))

# Menampilkan contoh hasil alignment
display(sales_2025_visual.head(10))

## 6. Mapping Summary

In [ ]:
# ==========================================
# 7. 6. MAPPING SUMMARY
# ==========================================
# Meringkas pasangan SKU Real ke SKU Edit agar perubahan alignment mudah diaudit.

mapping_summary = (
    sales_2025_visual[sales_2025_visual["Real != Edit"]]
    .groupby(["Group Key", "SKU Real", "SKU Edit"], as_index=False)
    .agg(
        rows=("CF Real", "size"),
        cf_real=("CF Real", "sum"),
        cf_edit=("CF Edit", "sum"),
        cf_adjustment=("CF Adjustment", lambda s: ", ".join(sorted(set(s)))),
        first_date=("Date", "min"),
        last_date=("Date", "max"),
    )
    .sort_values(["Group Key", "SKU Real"])
)

display(mapping_summary)

## 7. Sales 2026 Alignment

In [ ]:
# ==========================================
# 8. 7. SALES 2026 ALIGNMENT
# ==========================================
# Menerapkan rule alignment yang sama pada Sales 2026 agar data 2025 dan 2026 konsisten.

# Membentuk dataframe Sales 2026 dengan nama kolom yang disamakan dengan Sales 2025
actual26 = pd.DataFrame({
    "Date": pd.to_datetime(sales26["fecha_liquidacion"], errors="coerce"),
    "Channel Group": sales26["DescCanalLocal (grupo)"],
    "Branch": sales26["desc_sucursal"],
    "Channel": sales26["DescCanalLocal"],
    "Cust Code": sales26["cod_cliente"].map(clean_code),
    "Customer Name": sales26["nomb_cliente"],
    "Brand": sales26["desc_marca"].map(clean_text),
    "Flavor": sales26["desc_sabor"].map(clean_text),
    "Format": pd.to_numeric(sales26["desc_formato"], errors="coerce"),
    "Box Content": sales26["cant_contenido"].map(clean_code),
    "CF": pd.to_numeric(sales26["CF"], errors="coerce").fillna(0),
    "Item Code Real": sales26["cod_articulo"].map(clean_code),
    "Short Item Description Real": sales26["desc_articulo_corto"].map(clean_text),
})

# Menghapus baris tanpa tanggal dan membuat metadata bulan
actual26 = actual26.dropna(subset=["Date"]).copy()
actual26["Month"] = actual26["Date"].dt.to_period("M").dt.to_timestamp()

# Membuat Group Key dan SKU Real sebelum alignment
actual26["Group Key"] = [group_key(b, f, s) for b, f, s in zip(actual26["Brand"], actual26["Flavor"], actual26["Format"])]
actual26["SKU Real"] = actual26["Item Code Real"] + " | " + actual26["Short Item Description Real"]

# Menggabungkan Sales 2026 dengan referensi SKU terbaru
aligned26 = actual26.merge(sku_reference, on=["Brand", "Flavor", "Group Key"], how="left")

# Syarat acuan valid untuk update SKU 2026
has_reference26 = aligned26["Group Key"].ne("") & aligned26["Item Code Edit"].notna()

# Mengisi kode dan deskripsi edit dari referensi jika tersedia
aligned26["Item Code Edit"] = np.where(has_reference26, aligned26["Item Code Edit"], aligned26["Item Code Real"])
aligned26["Short Item Description Edit"] = np.where(
    has_reference26,
    aligned26["Short Item Description Edit"],
    aligned26["Short Item Description Real"],
)

# Membentuk label SKU Edit dan flag perubahan Real vs Edit
aligned26["SKU Edit"] = aligned26["Item Code Edit"].map(clean_code) + " | " + aligned26["Short Item Description Edit"].map(clean_text)
aligned26["Real != Edit"] = aligned26["SKU Real"] != aligned26["SKU Edit"]

# Menyimpan CF asli dan menerapkan adjustment CF / 2 untuk pack 12 ke 24
aligned26["CF Real"] = aligned26["CF"]
pack_12_to_24_26 = aligned26["Box Content"].eq("12") & aligned26["Reference Box Content"].eq("24") & aligned26["Real != Edit"]
aligned26["CF Edit"] = np.where(pack_12_to_24_26, aligned26["CF Real"] / 2, aligned26["CF Real"])
aligned26["CF Adjustment"] = np.where(pack_12_to_24_26, "12-pack to 24-pack: CF / 2", "No CF adjustment")

# Memilih kolom output yang sama dengan Sales 2025 agar bisa di-append
sales_2026_visual = aligned26[[
    "Date", "Channel Group", "Branch", "Channel", "Cust Code", "Customer Name",
    "Short Item Description Real", "Item Code Real",
    "Short Item Description Edit", "Item Code Edit",
    "Format", "Box Content", "Reference Box Content",
    "CF Real", "CF Edit", "CF Adjustment",
    "Group Key", "SKU Real", "SKU Edit", "Real != Edit",
]].copy()

# Menambahkan identitas sumber dan metadata waktu
sales_2026_visual.insert(0, "Source Sheet", "Sales 2026")
sales_2026_visual.insert(1, "Year", sales_2026_visual["Date"].dt.year)
sales_2026_visual.insert(2, "Month No", sales_2026_visual["Date"].dt.month)
sales_2026_visual["Month"] = sales_2026_visual["Date"].dt.to_period("M").dt.to_timestamp()

# Membuat ringkasan mapping SKU 2026 yang berubah
mapping_summary_2026 = (
    sales_2026_visual[sales_2026_visual["Real != Edit"]]
    .groupby(["Group Key", "SKU Real", "SKU Edit"], as_index=False)
    .agg(
        rows=("CF Real", "size"),
        cf_real=("CF Real", "sum"),
        cf_edit=("CF Edit", "sum"),
        cf_adjustment=("CF Adjustment", lambda s: ", ".join(sorted(set(s)))),
        first_date=("Date", "min"),
        last_date=("Date", "max"),
    )
    .sort_values(["Group Key", "SKU Real"])
)

# Menampilkan ringkasan hasil alignment Sales 2026
print("Sales 2026 rows:", len(sales_2026_visual))
print("Sales 2026 rows changed Real -> Edit:", int(sales_2026_visual["Real != Edit"].sum()))
print("Sales 2026 rows with CF / 2 adjustment:", int((sales_2026_visual["CF Adjustment"] == "12-pack to 24-pack: CF / 2").sum()))
display(mapping_summary_2026)

## 8. Append Sales 2025 + Sales 2026

In [ ]:
# ==========================================
# 9. 8. APPEND SALES 2025 + SALES 2026
# ==========================================
# Menggabungkan data aligned 2025 dan 2026 menjadi satu dataset analisis.

append_columns = [
    "Source Sheet", "Year", "Month No", "Date",
    "Channel Group", "Branch", "Channel", "Cust Code", "Customer Name",
    "Short Item Description Real", "Item Code Real",
    "Short Item Description Edit", "Item Code Edit",
    "Format", "Box Content", "Reference Box Content",
    "CF Real", "CF Edit", "CF Adjustment",
    "Group Key", "SKU Real", "SKU Edit", "Real != Edit",
]

sales_aligned_append = pd.concat(
    [
        sales_2025_visual[append_columns],
        sales_2026_visual[append_columns],
    ],
    ignore_index=True,
)
sales_aligned_append["Month"] = sales_aligned_append["Date"].dt.to_period("M").dt.to_timestamp()

print("Appended rows:", len(sales_aligned_append))
print("Sales 2025 rows:", len(sales_2025_visual))
print("Sales 2026 rows:", len(sales_2026_visual))
print("Total rows changed Real -> Edit:", int(sales_aligned_append["Real != Edit"].sum()))
display(sales_aligned_append.head(10))

## 9. VOLT CF / 2 Check - Real vs Edit

In [ ]:
# ==========================================
# 10. 9. VOLT CF / 2 CHECK - REAL VS EDIT
# ==========================================
# Memvisualisasikan VOLT sebelum dan sesudah CF dibagi 2 untuk audit pack 12 ke 24.

keyword = "VOLT"

filtered = sales_aligned_append[
    sales_aligned_append["SKU Real"].str.contains(keyword, case=False, na=False)
    | sales_aligned_append["SKU Edit"].str.contains(keyword, case=False, na=False)
].copy()

filtered_long = pd.concat(
    [
        filtered.assign(SKU_View="Real Before CF / 2", SKU=filtered["SKU Real"], CF_View=filtered["CF Real"]),
        filtered.assign(SKU_View="Edit After CF / 2", SKU=filtered["SKU Edit"], CF_View=filtered["CF Edit"]),
    ],
    ignore_index=True,
)

filtered_monthly = (
    filtered_long
    .groupby(["Month", "SKU_View", "SKU"], as_index=False)
    .agg(CF=("CF_View", "sum"))
)

display(filtered_monthly)

if px:
    fig = px.line(
        filtered_monthly,
        x="Month",
        y="CF",
        color="SKU",
        line_dash="SKU_View",
        markers=True,
        category_orders={"SKU_View": ["Real Before CF / 2", "Edit After CF / 2"]},
        title=f"{keyword}: Real Before CF / 2 vs Edit After CF / 2",
    )
    fig.show()

## 10. Appended Monthly CF by SKU - Edit

In [ ]:
# ==========================================
# 11. 10. APPENDED MONTHLY CF BY SKU - EDIT
# ==========================================
# Membuat line chart bulanan berdasarkan SKU Edit setelah data 2025 dan 2026 digabung.

append_edit_monthly = sales_aligned_append.groupby(["Month", "SKU Edit"], as_index=False).agg(CF=("CF Edit", "sum"))
top_append_edit = append_edit_monthly.groupby("SKU Edit")["CF"].sum().nlargest(20).index
append_edit_top = append_edit_monthly[append_edit_monthly["SKU Edit"].isin(top_append_edit)]

if px:
    fig = px.line(append_edit_top, x="Month", y="CF", color="SKU Edit", markers=True, title="Sales 2025-2026 Monthly CF by SKU - Edit Top 20")
    fig.show()
else:
    display(append_edit_top.head(50))

## 11. Sales 2026 Monthly CF by SKU - Edit

In [ ]:
# ==========================================
# 12. 11. SALES 2026 MONTHLY CF BY SKU - EDIT
# ==========================================
# Mengecek tren 2026 saja setelah alignment SKU.

edit_monthly_2026 = sales_2026_visual.groupby(["Month", "SKU Edit"], as_index=False).agg(CF=("CF Edit", "sum"))
top_edit_2026 = edit_monthly_2026.groupby("SKU Edit")["CF"].sum().nlargest(20).index
edit_top_2026 = edit_monthly_2026[edit_monthly_2026["SKU Edit"].isin(top_edit_2026)]

if px:
    fig = px.line(edit_top_2026, x="Month", y="CF", color="SKU Edit", markers=True, title="Sales 2026 Monthly CF by SKU - Edit Top 20")
    fig.show()
else:
    display(edit_top_2026.head(50))

## 12. Distinct SKU Count: Real vs Edit

In [ ]:
# ==========================================
# 13. 12. DISTINCT SKU COUNT: REAL VS EDIT
# ==========================================
# Membandingkan jumlah SKU unik sebelum dan sesudah alignment per bulan.

distinct_monthly = (
    sales_aligned_append.groupby("Month")
    .agg(
        real_sku_count=("SKU Real", "nunique"),
        edit_sku_count=("SKU Edit", "nunique"),
        cf_real=("CF Real", "sum"),
        cf_edit=("CF Edit", "sum"),
    )
    .reset_index()
)

display(distinct_monthly)

if px:
    fig = px.line(
        distinct_monthly,
        x="Month",
        y=["real_sku_count", "edit_sku_count"],
        markers=True,
        title="Distinct SKU Count per Month: Real vs Edit - Sales 2025-2026",
    )
    fig.show()

## 13. Monthly CF by SKU - Real

In [ ]:
# ==========================================
# 14. 13. MONTHLY CF BY SKU - REAL
# ==========================================
# Menampilkan trend bulanan berdasarkan SKU Real sebagai pembanding sebelum edit.

real_monthly = sales_aligned_append.groupby(["Month", "SKU Real"], as_index=False).agg(CF=("CF Real", "sum"))
top_real = real_monthly.groupby("SKU Real")["CF"].sum().nlargest(20).index
real_top = real_monthly[real_monthly["SKU Real"].isin(top_real)]

if px:
    fig = px.line(real_top, x="Month", y="CF", color="SKU Real", markers=True, title="Sales 2025-2026 Monthly CF by SKU - Real Top 20")
    fig.show()
else:
    display(real_top.head(50))

## 14. Monthly CF by SKU - Edit

In [ ]:
# ==========================================
# 15. 14. MONTHLY CF BY SKU - EDIT
# ==========================================
# Menampilkan trend bulanan berdasarkan SKU Edit sebagai hasil setelah alignment.

edit_monthly = sales_aligned_append.groupby(["Month", "SKU Edit"], as_index=False).agg(CF=("CF Edit", "sum"))
top_edit = edit_monthly.groupby("SKU Edit")["CF"].sum().nlargest(20).index
edit_top = edit_monthly[edit_monthly["SKU Edit"].isin(top_edit)]

if px:
    fig = px.line(edit_top, x="Month", y="CF", color="SKU Edit", markers=True, title="Sales 2025-2026 Monthly CF by SKU - Edit Top 20")
    fig.show()
else:
    display(edit_top.head(50))

## 15. Sep 2026 Forecast Detail

Forecast ini memakai grain `Cust Code x SKU`, dengan baseline yang dibersihkan dari suspected promo/outlier secara konservatif.
Sebelum forecast, `Customer Name` dan `Branch` diselaraskan dulu agar 1 Cust Code hanya punya 1 nama dan 1 branch utama.

Parameter utama:
- `FORECAST_MONTH`: bulan yang akan diforecast
- `RUNRATE_MONTH`: bulan berjalan yang masih MTD
- `CF Edit`: volume yang sudah disetarakan SKU dan pack
- Forecast engine difilter hanya untuk `Channel Group = MODERN`
- Jika 1 Cust Code muncul dengan beberapa nama atau branch, pakai nama dan branch dengan jumlah transaksi/order row terbanyak
- Agustus run-rate memakai multiplier konservatif `1.5`
- Juli dianggap promo dan tidak menjadi driver utama baseline September

In [ ]:
# ==========================================
# 16. 15. SEP 2026 FORECAST DETAIL
# ==========================================
# Menyiapkan data forecast MODERN, parameter run-rate, dan grain Customer x SKU.

# Menetapkan bulan forecast, bulan run-rate, multiplier run-rate, dan scope channel
FORECAST_MONTH = pd.Timestamp("2026-09-01")
RUNRATE_MONTH = FORECAST_MONTH - pd.DateOffset(months=1)
RUNRATE_MULTIPLIER = 1.5
FORECAST_CHANNEL_GROUP = "MODERN"
KNOWN_PROMO_MONTHS = [pd.Timestamp("2026-07-01")]

# Grain forecast: Cust Code x SKU; Customer Name dan Branch sudah dinormalisasi sebagai label utama
key_cols = [
    "Channel Group", "Branch", "Cust Code", "Customer Name",
    "Item Code Edit", "Short Item Description Edit", "SKU Edit",
]

# Filter forecast hanya untuk Channel Group MODERN sesuai scope analisis
forecast_source = sales_aligned_append[
    sales_aligned_append["Channel Group"].map(clean_text).eq(FORECAST_CHANNEL_GROUP)
].copy()

# ==========================================
# 1. NORMALISASI IDENTITAS CUSTOMER
# ==========================================
# Menyimpan nama customer dan branch asli sebelum dibersihkan agar export order tetap bisa diaudit
forecast_source["Original Customer Name"] = forecast_source["Customer Name"]
forecast_source["Original Branch"] = forecast_source["Branch"]

# Membuat acuan nama utama per Cust Code berdasarkan jumlah baris transaksi terbanyak
customer_name_reference = (
    forecast_source
    .groupby(["Cust Code", "Customer Name"], as_index=False)
    .agg(
        order_rows=("CF Edit", "size"),
        cf_edit=("CF Edit", "sum"),
        last_order_date=("Date", "max"),
    )
    .sort_values(
        ["Cust Code", "order_rows", "cf_edit", "last_order_date", "Customer Name"],
        ascending=[True, False, False, False, True],
    )
    .drop_duplicates(["Cust Code"], keep="first")
    .rename(columns={"Customer Name": "Primary Customer Name"})
)

# Membuat acuan branch utama per Cust Code berdasarkan jumlah baris transaksi terbanyak
customer_branch_reference = (
    forecast_source
    .groupby(["Cust Code", "Branch"], as_index=False)
    .agg(
        order_rows=("CF Edit", "size"),
        cf_edit=("CF Edit", "sum"),
        last_order_date=("Date", "max"),
    )
    .sort_values(
        ["Cust Code", "order_rows", "cf_edit", "last_order_date", "Branch"],
        ascending=[True, False, False, False, True],
    )
    .drop_duplicates(["Cust Code"], keep="first")
    .rename(columns={"Branch": "Primary Branch"})
)

# Menampilkan Cust Code yang sebelumnya muncul dengan lebih dari 1 nama sebagai audit pembersihan
multi_name_customer_audit = (
    forecast_source
    .groupby(["Cust Code"], as_index=False)
    .agg(customer_name_count=("Customer Name", "nunique"))
    .query("customer_name_count > 1")
    .merge(
        customer_name_reference[["Cust Code", "Primary Customer Name", "order_rows", "cf_edit"]],
        on="Cust Code",
        how="left",
    )
    .sort_values(["customer_name_count", "cf_edit"], ascending=[False, False])
)

# Menampilkan Cust Code yang sebelumnya muncul di lebih dari 1 branch sebagai audit pembersihan
multi_branch_customer_audit = (
    forecast_source
    .groupby(["Cust Code"], as_index=False)
    .agg(branch_count=("Branch", "nunique"))
    .query("branch_count > 1")
    .merge(
        customer_branch_reference[["Cust Code", "Primary Branch", "order_rows", "cf_edit"]],
        on="Cust Code",
        how="left",
    )
    .sort_values(["branch_count", "cf_edit"], ascending=[False, False])
)

# Mengganti Customer Name dan Branch menjadi label utama supaya forecast benar-benar Cust Code x SKU
forecast_source = forecast_source.merge(
    customer_name_reference[["Cust Code", "Primary Customer Name"]],
    on="Cust Code",
    how="left",
)
forecast_source = forecast_source.merge(
    customer_branch_reference[["Cust Code", "Primary Branch"]],
    on="Cust Code",
    how="left",
)
forecast_source["Customer Name"] = forecast_source["Primary Customer Name"].fillna(forecast_source["Customer Name"])
forecast_source["Branch"] = forecast_source["Primary Branch"].fillna(forecast_source["Branch"])
forecast_source = forecast_source.drop(columns=["Primary Customer Name", "Primary Branch"])

# ==========================================
# 2. AGREGASI HARIAN CUSTOMER X SKU
# ==========================================
# Agregasi harian agar data bisa dicek dari tanggal transaksi sebelum menjadi bulanan
daily_customer_sku = (
    forecast_source
    .groupby(["Date", "Month", *key_cols], as_index=False)
    .agg(CF=("CF Edit", "sum"))
)

# Mengecek tanggal data terakhir untuk membaca status bulan run-rate
max_data_date = daily_customer_sku["Date"].max()
if max_data_date.to_period("M").to_timestamp() == RUNRATE_MONTH:
    elapsed_days = max_data_date.day
else:
    elapsed_days = daily_customer_sku.loc[daily_customer_sku["Month"].eq(RUNRATE_MONTH), "Date"].dt.day.max()
    elapsed_days = 0 if pd.isna(elapsed_days) else int(elapsed_days)

# Menggunakan multiplier tetap 1.5, bukan run-rate kalender
runrate_multiplier = RUNRATE_MULTIPLIER

# Agregasi bulanan pada grain Customer x SKU sebagai input forecast
monthly_customer_sku = (
    daily_customer_sku
    .groupby(["Month", *key_cols], as_index=False)
    .agg(CF=("CF", "sum"))
)

# Menampilkan parameter utama agar user bisa audit scope forecast sebelum melihat angka
print("Forecast month:", FORECAST_MONTH.date())
print("Forecast channel group:", FORECAST_CHANNEL_GROUP)
print("Run-rate month:", RUNRATE_MONTH.date())
print("Max data date:", max_data_date.date())
print("Run-rate elapsed days:", elapsed_days)
print("Run-rate multiplier:", round(runrate_multiplier, 4), "(fixed conservative multiplier)")
print("Known promo months excluded from Sep baseline:", [m.strftime("%b %Y") for m in KNOWN_PROMO_MONTHS])
print("Forecast source rows:", len(forecast_source))
print("Customers with multiple names before cleanup:", len(multi_name_customer_audit))
print("Customers with multiple branches before cleanup:", len(multi_branch_customer_audit))
display(multi_name_customer_audit.head(30))
display(multi_branch_customer_audit.head(30))
display(monthly_customer_sku.head())

## 16. Conservative Promo/Outlier Cleaning

In [ ]:
# ==========================================
# 17. 16. CONSERVATIVE PROMO/OUTLIER CLEANING
# ==========================================
# Membersihkan baseline dari promo/outlier secara konservatif dan menghitung Aug run-rate.

# Bulan input forecast dimulai dari Mar sampai bulan run-rate; Jan-Feb hanya dipakai sebagai history cleaning
forecast_input_months = pd.date_range("2026-03-01", RUNRATE_MONTH, freq="MS")
history_months = pd.date_range("2026-01-01", RUNRATE_MONTH, freq="MS")

# Membentuk data lebar: satu baris Customer x SKU, kolomnya bulan-bulan history
wide = monthly_customer_sku.pivot_table(
    index=key_cols,
    columns="Month",
    values="CF",
    aggfunc="sum",
    fill_value=0,
).reset_index()

# Memastikan semua bulan history tersedia sebagai kolom meskipun nilainya 0
for month in history_months:
    if month not in wide.columns:
        wide[month] = 0

wide = wide[key_cols + list(history_months)]

# Fungsi untuk membersihkan promo/outlier per baris Customer x SKU
def clean_months(row):
    values = {m: float(row[m]) for m in history_months}
    clean = {}
    promo_flags = {}
    promo_uplift = {}

    for i, month in enumerate(history_months):
        actual = values[month]
        previous = [clean[m] for m in history_months[max(0, i - 3):i] if clean.get(m, 0) > 0]

        # Jika bulan termasuk known promo, baseline dibatasi ke median history sebelumnya
        if month in KNOWN_PROMO_MONTHS and actual > 0:
            if previous:
                med = float(np.median(previous))
                is_promo = True
                clean_value = min(actual, med)
            else:
                is_promo = True
                clean_value = actual

        # Jika bukan known promo, deteksi outlier secara konservatif menggunakan median dan standar deviasi
        elif len(previous) >= 2:
            med = float(np.median(previous))
            std = float(np.std(previous))
            threshold = max(med * 2.0, med + 2.0 * std)
            is_promo = med > 0 and actual >= 10 and actual > threshold
            clean_value = min(actual, med) if is_promo else actual
        else:
            is_promo = False
            clean_value = actual
        clean[month] = clean_value
        promo_flags[month] = is_promo
        promo_uplift[month] = max(0, actual - clean_value)

    # Agustus dianggap MTD dan di-run-rate dengan multiplier tetap 1.5
    aug_actual = values.get(RUNRATE_MONTH, 0)
    aug_clean_mtd = clean.get(RUNRATE_MONTH, 0)
    aug_rr = aug_actual * runrate_multiplier
    aug_clean_rr = aug_clean_mtd * runrate_multiplier

    # Mengembalikan kolom actual, clean, run-rate, dan dampak promo/outlier
    return pd.Series({
        **{f"{m.strftime('%b')}_Actual": values[m] for m in forecast_input_months},
        **{f"{m.strftime('%b')}_Clean": (aug_clean_rr if m == RUNRATE_MONTH else clean[m]) for m in forecast_input_months},
        "Aug_MTD": aug_actual,
        "Aug_RunRate": aug_rr,
        "Aug_Clean_RunRate": aug_clean_rr,
        "Promo_Suspect_Months": int(sum(promo_flags.values())),
        "Promo_Suspect_Uplift": float(sum(promo_uplift.values())),
    })

clean_features = wide.apply(clean_months, axis=1)
forecast_base = pd.concat([wide[key_cols].reset_index(drop=True), clean_features], axis=1)

display(forecast_base.head())

## 17. Segmented Robust Forecast

In [ ]:
# ==========================================
# 18. 17. SEGMENTED ROBUST FORECAST
# ==========================================
# Mengklasifikasikan demand dan menghitung forecast low/base/high September.

non_promo_cols = ["Mar_Clean", "Apr_Clean", "May_Clean", "Jun_Clean", "Aug_Clean"]
recent_non_promo_cols = ["Jun_Clean", "Aug_Clean"]
inactive_3m_cols = ["Jun_Actual", "Jul_Actual", "Aug_Actual"]

def nonzero(values):
    return [float(v) for v in values if float(v) > 0]

def trimmed_mean(values):
    vals = [float(v) for v in values]
    if len(vals) >= 4:
        vals = sorted(vals)[1:-1]
    return float(np.mean(vals)) if vals else 0.0

def classify_and_forecast(row):
    non_promo = [row[c] for c in non_promo_cols]
    recent_non_promo = [row[c] for c in recent_non_promo_cols]
    inactive_3m = all(float(row[c]) == 0 for c in inactive_3m_cols)
    nz = nonzero(non_promo)
    active_months = len(nz)
    total_recent = float(sum(non_promo))
    avg = float(np.mean(nz)) if nz else 0.0
    volatility = float(np.std(nz) / avg) if avg > 0 and len(nz) >= 2 else 0.0

    median_recent = float(np.median(nonzero(recent_non_promo))) if nonzero(recent_non_promo) else 0.0
    median_non_promo = float(np.median(nz)) if nz else 0.0
    recent_non_promo_avg = float(0.50 * row["Jun_Clean"] + 0.50 * row["Aug_Clean"])
    trimmed_non_promo = trimmed_mean(non_promo)

    # Jika 3 bulan kalender terakhir tidak ada order aktual, forecast September dipaksa 0
    if inactive_3m:
        demand_class = "Inactive 3M"
        raw_fc = 0.0
    elif total_recent == 0:
        demand_class = "Inactive"
        raw_fc = 0.0
    elif active_months <= 2:
        demand_class = "New / Sparse"
        raw_fc = max(median_non_promo, row["Aug_Clean"] * 0.60)
    elif active_months <= 3 or volatility > 1.50:
        demand_class = "Intermittent"
        raw_fc = median_non_promo
    elif volatility > 0.75:
        demand_class = "Volatile"
        raw_fc = 0.60 * median_non_promo + 0.40 * trimmed_non_promo
    else:
        demand_class = "Stable"
        raw_fc = 0.50 * recent_non_promo_avg + 0.50 * median_non_promo

    max_non_promo = max(non_promo) if non_promo else 0.0
    upper_guardrail = max_non_promo * 1.20 if max_non_promo > 0 else raw_fc
    guarded_fc = min(max(raw_fc, 0.0), upper_guardrail)

    # Jika Agustus clean = 0, forecast tidak boleh lebih tinggi dari sinyal recent non-promo
    # Contoh: masih ada order Jun/Jul, tetapi tidak ada order Agustus; jangan pakai full median history lama
    current_month_guardrail = float(row["Aug_Clean"]) == 0 and guarded_fc > median_recent
    base_fc = min(guarded_fc, median_recent) if current_month_guardrail else guarded_fc
    low_fc = base_fc * 0.90
    high_fc = base_fc * 1.15

    return pd.Series({
        "Active Months": active_months,
        "Inactive 3M Rule": inactive_3m,
        "Volatility": volatility,
        "Demand Class": demand_class,
        "Median Recent Non-Promo": median_recent,
        "Median Non-Promo": median_non_promo,
        "Recent Non-Promo Avg": recent_non_promo_avg,
        "Trimmed Mean Non-Promo": trimmed_non_promo,
        "Raw Forecast": raw_fc,
        "Guardrail Cap": upper_guardrail,
        "Current Month Guardrail": current_month_guardrail,
        "Sep Forecast Base": base_fc,
        "Sep Forecast Low": low_fc,
        "Sep Forecast High": high_fc,
    })

forecast_metrics = forecast_base.apply(classify_and_forecast, axis=1)
forecast_detail_sep = pd.concat([forecast_base, forecast_metrics], axis=1)
forecast_detail_sep = forecast_detail_sep.sort_values("Sep Forecast Base", ascending=False)

display(forecast_detail_sep.head(30))

## 18. Inactive 3M Customer Matrix

Bagian ini menampilkan `Customer x SKU` yang terkena rule:
`Jun_Actual = 0`, `Jul_Actual = 0`, dan `Aug_Actual = 0`.

Untuk baris ini, forecast September dipaksa menjadi `0`.

In [ ]:
# ==========================================
# 19. 18. INACTIVE 3M CUSTOMER MATRIX
# ==========================================
# Menampilkan Customer x SKU yang tidak order tiga bulan terakhir dalam matrix bulanan 2026.

# Mengambil baris Customer x SKU yang terkena rule tidak order 3 bulan terakhir
inactive_3m_detail = (
    forecast_detail_sep[forecast_detail_sep["Inactive 3M Rule"]]
    .copy()
    .sort_values(["Customer Name", "SKU Edit"])
)

# Menyiapkan daftar bulan setahun penuh untuk matrix 2026
matrix_months = pd.date_range("2026-01-01", "2026-12-01", freq="MS")
matrix_month_labels = [m.strftime("%b_Actual") for m in matrix_months]

# Mengambil actual bulanan 2026 pada grain Customer x SKU
monthly_actual_2026 = monthly_customer_sku[
    monthly_customer_sku["Month"].dt.year.eq(2026)
].copy()

# Membuat matrix actual order per Customer x SKU selama 2026
inactive_3m_matrix = (
    inactive_3m_detail[key_cols + ["Inactive 3M Rule", "Demand Class", "Sep Forecast Base"]]
    .merge(monthly_actual_2026, on=key_cols, how="left")
    .pivot_table(
        index=key_cols + ["Inactive 3M Rule", "Demand Class", "Sep Forecast Base"],
        columns="Month",
        values="CF",
        aggfunc="sum",
        fill_value=0,
    )
    .reset_index()
)

# Menambahkan kolom bulan yang belum ada agar matrix tetap Jan-Dec
for month in matrix_months:
    if month not in inactive_3m_matrix.columns:
        inactive_3m_matrix[month] = 0

# Rename kolom tanggal menjadi nama bulan yang mudah dibaca
inactive_3m_matrix = inactive_3m_matrix[
    key_cols + ["Inactive 3M Rule", "Demand Class", "Sep Forecast Base"] + list(matrix_months)
].rename(columns={month: label for month, label in zip(matrix_months, matrix_month_labels)})

# Menambahkan total actual Jan-Aug sebagai indikator skala history sebelum inactive
actual_cols_to_date = [m.strftime("%b_Actual") for m in pd.date_range("2026-01-01", RUNRATE_MONTH, freq="MS")]
inactive_3m_matrix["Actual YTD"] = inactive_3m_matrix[actual_cols_to_date].sum(axis=1)
inactive_3m_matrix = inactive_3m_matrix.sort_values("Actual YTD", ascending=False)

print("Inactive 3M rows:", len(inactive_3m_detail))
display(inactive_3m_detail.head(30))
display(inactive_3m_matrix.head(50))

if px and len(inactive_3m_matrix) > 0:
    heatmap_source = inactive_3m_matrix.head(50).melt(
        id_vars=["Customer Name", "SKU Edit"],
        value_vars=matrix_month_labels,
        var_name="Month",
        value_name="Actual CF",
    )
    heatmap_source["Customer x SKU"] = heatmap_source["Customer Name"] + " | " + heatmap_source["SKU Edit"]
    heatmap_matrix = heatmap_source.pivot_table(
        index="Customer x SKU",
        columns="Month",
        values="Actual CF",
        aggfunc="sum",
        fill_value=0,
    )

    fig = px.imshow(
        heatmap_matrix,
        aspect="auto",
        title="Inactive 3M Customer x SKU Matrix - Actual CF 2026 Top 50 by YTD",
    )
    fig.update_layout(xaxis_title="Month", yaxis_title="Customer x SKU")
    fig.show()

## 19. Forecast Process Visualization

Bagian ini dibuat untuk audit:
- memastikan forecast sudah hanya `MODERN`
- melihat perubahan `Actual` menjadi `Clean`
- melihat dampak run-rate Agustus x1.5
- membedakan Agustus MTD vs Agustus run-rate yang dipakai forecast
- melihat area yang paling banyak terkena promo/outlier cleaning
- melihat distribusi hasil forecast

In [ ]:
# ==========================================
# 20. 19. FORECAST PROCESS VISUALIZATION
# ==========================================
# Membuat visual audit untuk scope, cleaning, run-rate, class, branch, SKU, dan customer.

scope_months = pd.date_range("2026-01-01", RUNRATE_MONTH, freq="MS")

channel_scope = (
    sales_aligned_append[sales_aligned_append["Month"].isin(scope_months)]
    .assign(Channel_Group_Clean=lambda d: d["Channel Group"].map(clean_text))
    .groupby("Channel_Group_Clean", as_index=False)
    .agg(rows=("CF Edit", "size"), cf_edit=("CF Edit", "sum"))
    .sort_values("cf_edit", ascending=False)
)

monthly_process = []
for month in forecast_input_months:
    mon = month.strftime("%b")
    actual_cf = forecast_detail_sep[f"{mon}_Actual"].sum()
    clean_cf = forecast_detail_sep[f"{mon}_Clean"].sum()
    monthly_process.append({
        "Month": mon,
        "Metric": "Actual Full Month / MTD",
        "CF": actual_cf,
        "Note": "Actual MTD" if month == RUNRATE_MONTH else "Actual full month",
    })
    if month == RUNRATE_MONTH:
        monthly_process.append({
            "Month": mon,
            "Metric": "Actual Run-rate x1.5",
            "CF": forecast_detail_sep["Aug_RunRate"].sum(),
            "Note": "Actual MTD multiplied by fixed 1.5 run-rate",
        })
    monthly_process.append({
        "Month": mon,
        "Metric": "Clean Baseline Used",
        "CF": clean_cf,
        "Note": "Clean MTD x 1.5, used by forecast" if month == RUNRATE_MONTH else "Clean full month, used by forecast",
    })
monthly_process = pd.DataFrame(monthly_process)

aug_diagnostic = pd.DataFrame({
    "Metric": [
        "Aug Actual MTD",
        "Aug Actual Run-rate x1.5",
        "Aug Clean Baseline Used",
    ],
    "CF": [
        forecast_detail_sep["Aug_MTD"].sum(),
        forecast_detail_sep["Aug_RunRate"].sum(),
        forecast_detail_sep["Aug_Clean_RunRate"].sum(),
    ],
})

promo_cleaning_by_sku = (
    forecast_detail_sep
    .groupby("SKU Edit", as_index=False)
    .agg(
        promo_uplift=("Promo_Suspect_Uplift", "sum"),
        promo_months=("Promo_Suspect_Months", "sum"),
        sep_base=("Sep Forecast Base", "sum"),
    )
    .sort_values("promo_uplift", ascending=False)
)

viz_by_class = (
    forecast_detail_sep
    .groupby("Demand Class", as_index=False)
    .agg(
        rows=("SKU Edit", "size"),
        sep_base=("Sep Forecast Base", "sum"),
        sep_low=("Sep Forecast Low", "sum"),
        sep_high=("Sep Forecast High", "sum"),
    )
    .sort_values("sep_base", ascending=False)
)

viz_by_branch = (
    forecast_detail_sep
    .groupby("Branch", as_index=False)
    .agg(sep_base=("Sep Forecast Base", "sum"), rows=("SKU Edit", "size"))
    .sort_values("sep_base", ascending=False)
)

viz_by_customer = (
    forecast_detail_sep
    .groupby(["Cust Code", "Customer Name"], as_index=False)
    .agg(sep_base=("Sep Forecast Base", "sum"), rows=("SKU Edit", "size"))
    .sort_values("sep_base", ascending=False)
)

viz_by_sku = (
    forecast_detail_sep
    .groupby("SKU Edit", as_index=False)
    .agg(sep_base=("Sep Forecast Base", "sum"), rows=("Cust Code", "size"))
    .sort_values("sep_base", ascending=False)
)

display(channel_scope)
display(aug_diagnostic)
display(monthly_process)
display(promo_cleaning_by_sku.head(20))

if px:
    fig = px.bar(
        channel_scope,
        x="Channel_Group_Clean",
        y="cf_edit",
        text="rows",
        title="Forecast Scope Check - Jan to Aug 2026 CF Edit by Channel Group",
    )
    fig.update_layout(xaxis_title="Channel Group", yaxis_title="CF Edit")
    fig.show()

    fig = px.bar(
        monthly_process,
        x="Month",
        y="CF",
        color="Metric",
        barmode="group",
        hover_data=["Note"],
        title="Forecast Baseline Process - Actual, Run-rate, and Clean Baseline",
        category_orders={"Month": [m.strftime("%b") for m in forecast_input_months]},
    )
    fig.update_layout(yaxis_title="CF")
    fig.show()

    fig = px.bar(
        promo_cleaning_by_sku.head(20),
        x="SKU Edit",
        y="promo_uplift",
        color="promo_months",
        title="Promo/Outlier Cleaning Impact by SKU - Top 20",
    )
    fig.update_layout(xaxis_title="SKU Edit", yaxis_title="CF Removed from Baseline")
    fig.show()

    fig = px.bar(
        viz_by_class,
        x="Demand Class",
        y="sep_base",
        text="rows",
        title="Sep 2026 Forecast by Demand Class",
    )
    fig.update_layout(yaxis_title="Sep Forecast Base")
    fig.show()

    fig = px.bar(
        viz_by_branch.head(20),
        x="Branch",
        y="sep_base",
        text="rows",
        title="Sep 2026 Forecast by Branch - Top 20",
    )
    fig.update_layout(yaxis_title="Sep Forecast Base")
    fig.show()

    fig = px.bar(
        viz_by_sku.head(20),
        x="SKU Edit",
        y="sep_base",
        text="rows",
        title="Sep 2026 Forecast by SKU - Top 20",
    )
    fig.update_layout(yaxis_title="Sep Forecast Base")
    fig.show()

    fig = px.bar(
        viz_by_customer.head(20),
        x="Customer Name",
        y="sep_base",
        text="rows",
        title="Sep 2026 Forecast by Customer - Top 20",
    )
    fig.update_layout(yaxis_title="Sep Forecast Base")
    fig.show()
else:
    display(viz_by_class)
    display(viz_by_branch.head(20))
    display(viz_by_sku.head(20))
    display(viz_by_customer.head(20))

## 20. Sep Forecast Summary

In [ ]:
# ==========================================
# 21. 20. SEP FORECAST SUMMARY
# ==========================================
# Meringkas hasil forecast September secara total, demand class, dan SKU.

summary_sep = pd.DataFrame({
    "Metric": [
        "Rows",
        "Sep Forecast Base",
        "Sep Forecast Low",
        "Sep Forecast High",
        "Promo Suspect Months",
        "Promo Suspect Uplift",
    ],
    "Value": [
        len(forecast_detail_sep),
        forecast_detail_sep["Sep Forecast Base"].sum(),
        forecast_detail_sep["Sep Forecast Low"].sum(),
        forecast_detail_sep["Sep Forecast High"].sum(),
        forecast_detail_sep["Promo_Suspect_Months"].sum(),
        forecast_detail_sep["Promo_Suspect_Uplift"].sum(),
    ],
})

summary_by_class_sep = (
    forecast_detail_sep
    .groupby("Demand Class", as_index=False)
    .agg(
        rows=("SKU Edit", "size"),
        sep_base=("Sep Forecast Base", "sum"),
        sep_low=("Sep Forecast Low", "sum"),
        sep_high=("Sep Forecast High", "sum"),
        promo_months=("Promo_Suspect_Months", "sum"),
    )
    .sort_values("sep_base", ascending=False)
)

summary_by_sku_sep = (
    forecast_detail_sep
    .groupby(["Item Code Edit", "Short Item Description Edit", "SKU Edit"], as_index=False)
    .agg(
        rows=("Cust Code", "size"),
        sep_base=("Sep Forecast Base", "sum"),
        sep_low=("Sep Forecast Low", "sum"),
        sep_high=("Sep Forecast High", "sum"),
    )
    .sort_values("sep_base", ascending=False)
)

display(summary_sep)
display(summary_by_class_sep)
display(summary_by_sku_sep.head(30))

if px:
    fig = px.bar(summary_by_sku_sep.head(20), x="SKU Edit", y="sep_base", title="Sep 2026 Forecast Base by SKU - Top 20")
    fig.show()

## 21. Monthly Forecast Chart

Chart ini menggabungkan history dan forecast dalam satu timeline:
- `Actual Full Month / MTD`: real sales yang tersedia
- `Clean Baseline Used`: baseline setelah promo/outlier cleaning dan Aug run-rate
- `Sep Forecast`: angka forecast September dengan low-high range

In [ ]:
# ==========================================
# 22. 21. MONTHLY FORECAST CHART
# ==========================================
# Menggabungkan history baseline dan forecast September dalam timeline bulanan.

forecast_monthly_total = pd.concat(
    [
        monthly_process[monthly_process["Metric"].isin(["Actual Full Month / MTD", "Clean Baseline Used"])]
        .assign(Month_Date=lambda d: pd.to_datetime("2026-" + d["Month"] + "-01", format="%Y-%b-%d"))
        [["Month_Date", "Metric", "CF"]],
        pd.DataFrame({
            "Month_Date": [FORECAST_MONTH, FORECAST_MONTH, FORECAST_MONTH],
            "Metric": ["Sep Forecast Low", "Sep Forecast Base", "Sep Forecast High"],
            "CF": [
                forecast_detail_sep["Sep Forecast Low"].sum(),
                forecast_detail_sep["Sep Forecast Base"].sum(),
                forecast_detail_sep["Sep Forecast High"].sum(),
            ],
        }),
    ],
    ignore_index=True,
)
forecast_monthly_total["Month Label"] = forecast_monthly_total["Month_Date"].dt.strftime("%b")

forecast_range_total = pd.DataFrame({
    "Month_Date": [FORECAST_MONTH],
    "Month Label": [FORECAST_MONTH.strftime("%b")],
    "Forecast Low": [forecast_detail_sep["Sep Forecast Low"].sum()],
    "Forecast Base": [forecast_detail_sep["Sep Forecast Base"].sum()],
    "Forecast High": [forecast_detail_sep["Sep Forecast High"].sum()],
})

sku_rank_cols = [f"{m.strftime('%b')}_Clean" for m in forecast_input_months]
sku_actual_rank_cols = [f"{m.strftime('%b')}_Actual" for m in forecast_input_months]
sku_history_rank = (
    forecast_detail_sep
    .assign(
        History_Actual=lambda d: d[sku_actual_rank_cols].sum(axis=1),
        History_Baseline=lambda d: d[sku_rank_cols].sum(axis=1),
    )
    .groupby("SKU Edit", as_index=False)
    .agg(
        history_actual=("History_Actual", "sum"),
        history_baseline=("History_Baseline", "sum"),
        sep_base=("Sep Forecast Base", "sum"),
    )
    .sort_values("history_actual", ascending=False)
)

sku_monthly_rows = []
top_history_sku = sku_history_rank.head(10)["SKU Edit"].tolist()

def add_monthly_row(rows, sku, month, metric, cf):
    rows.append({
        "SKU Edit": sku,
        "Month_Date": month,
        "Month Label": month.strftime("%b"),
        "Metric": metric,
        "CF": cf,
    })

for sku in top_history_sku:
    sku_rows = forecast_detail_sep[forecast_detail_sep["SKU Edit"].eq(sku)]
    for month in forecast_input_months:
        mon = month.strftime("%b")
        add_monthly_row(sku_monthly_rows, sku, month, "Clean Baseline Used", sku_rows[f"{mon}_Clean"].sum())
    add_monthly_row(sku_monthly_rows, sku, FORECAST_MONTH, "Sep Forecast Base", sku_rows["Sep Forecast Base"].sum())

sku_monthly_forecast = pd.DataFrame(sku_monthly_rows)

display(forecast_monthly_total)
display(forecast_range_total)
display(sku_history_rank.head(10))
display(sku_monthly_forecast.head(30))

# ==========================================
# 1. HELPER FORMAT CHART MATPLOTLIB
# ==========================================
# Membuat label angka lebih ringkas: 1,500 -> 1.5k
def fmt_k(value, _pos=None):
    value = float(value)
    if abs(value) >= 1000:
        return f"{value / 1000:.0f}k"
    return f"{value:.0f}"


# Memberi label angka di titik forecast agar low/base/high mudah dibaca
def label_point(ax, x, y, label, color, dx=8, dy=0):
    ax.annotate(
        f"{label}: {y:,.0f}",
        xy=(x, y),
        xytext=(dx, dy),
        textcoords="offset points",
        color=color,
        fontsize=9,
        fontweight="bold",
        va="center",
    )


# Memberi label angka untuk semua titik di satu series bulanan
def label_series_points(ax, data, x_col, y_col, color, dy=8, fontsize=8):
    for _, point in data.iterrows():
        ax.annotate(
            f"{float(point[y_col]):,.0f}",
            xy=(point[x_col], float(point[y_col])),
            xytext=(0, dy),
            textcoords="offset points",
            ha="center",
            va="bottom" if dy >= 0 else "top",
            fontsize=fontsize,
            color=color,
            fontweight="bold",
        )


# Memberi label untuk banyak SKU pada bulan yang sama dengan offset bertingkat agar tidak bertabrakan
def label_group_points_no_overlap(ax, data, color_map, x_col="Month_Date", y_col="CF", sku_col="SKU Edit"):
    for month, month_data in data.groupby(x_col):
        # Urutkan dari nilai terbesar supaya label tinggi mendapat offset paling atas
        month_data = month_data.sort_values(y_col, ascending=False).reset_index(drop=True)
        for rank, point in month_data.iterrows():
            value = float(point[y_col])
            if value == 0:
                continue
            offset = 12 + (rank % 5) * 11
            ax.annotate(
                f"{value:,.0f}",
                xy=(point[x_col], value),
                xytext=(0, offset),
                textcoords="offset points",
                ha="center",
                va="bottom",
                fontsize=7,
                color=color_map.get(point[sku_col], "#003049"),
                fontweight="bold",
                bbox=dict(boxstyle="round,pad=0.15", facecolor="white", edgecolor="none", alpha=0.75),
            )


# ==========================================
# 2. MONTHLY TIMELINE - BASELINE TERSAMBUNG KE FORECAST
# ==========================================
# Memisahkan actual dan clean baseline agar forecast tidak menjadi garis solid yang menyesatkan
actual_total = forecast_monthly_total[forecast_monthly_total["Metric"].eq("Actual Full Month / MTD")].sort_values("Month_Date")
baseline_total = forecast_monthly_total[forecast_monthly_total["Metric"].eq("Clean Baseline Used")].sort_values("Month_Date")

# Titik Agustus baseline menjadi anchor untuk garis putus-putus ke forecast September
aug_anchor_month = RUNRATE_MONTH
aug_anchor_value = float(baseline_total.loc[baseline_total["Month_Date"].eq(aug_anchor_month), "CF"].sum())
sep_low = float(forecast_detail_sep["Sep Forecast Low"].sum())
sep_base = float(forecast_detail_sep["Sep Forecast Base"].sum())
sep_high = float(forecast_detail_sep["Sep Forecast High"].sum())

if plt is not None:
    fig, ax = plt.subplots(figsize=(15, 5.5))
    ax.plot(actual_total["Month_Date"], actual_total["CF"], marker="o", linewidth=2.2, color="#4f63ff", label="Actual Full Month / MTD")
    ax.plot(baseline_total["Month_Date"], baseline_total["CF"], marker="o", linewidth=2.2, color="#ef4f3c", label="Clean Baseline Used")
    label_series_points(ax, actual_total, "Month_Date", "CF", "#4f63ff", dy=9)
    label_series_points(ax, baseline_total, "Month_Date", "CF", "#ef4f3c", dy=-14)

    forecast_points = [
        ("Low", sep_low, "#9b7cff"),
        ("Base", sep_base, "#00a676"),
        ("High", sep_high, "#ff9f43"),
    ]
    for label, value, color in forecast_points:
        ax.plot([aug_anchor_month, FORECAST_MONTH], [aug_anchor_value, value], linestyle="--", linewidth=2.2, color=color, alpha=0.95)
        ax.scatter([FORECAST_MONTH], [value], s=70, color=color, zorder=5, label=f"Sep Forecast {label}")
        label_point(ax, FORECAST_MONTH, value, label, color)

    ax.set_title("Monthly Timeline - Actual, Clean Baseline, and Sep Forecast", fontsize=15, fontweight="bold", loc="left")
    ax.set_xlabel("Month")
    ax.set_ylabel("CF")
    ax.yaxis.set_major_formatter(FuncFormatter(fmt_k))
    ax.grid(True, axis="y", alpha=0.25)
    ax.grid(True, axis="x", alpha=0.12)
    ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1.0), frameon=False)
    plt.tight_layout()
    plt.show()
else:
    print("Matplotlib is not available in this runtime; chart tables are displayed instead.")


# ==========================================
# 3. HISTORICAL SKU + FORECAST PER PACK SIZE
# ==========================================
# Membuat group pack size dari teks SKU agar chart mengikuti struktur seperti infografis
def pack_group_from_sku(sku):
    sku = clean_text(sku)
    if "VOLT" in sku or "200 ML" in sku or "200ML" in sku:
        return "VOLT / SINGLE SERVE"
    if "350ML" in sku or "350 ML" in sku:
        return "350 ML"
    if "400ML" in sku or "400 ML" in sku or " 400" in sku:
        return "400 ML"
    if "3LT" in sku or "3L" in sku or "3 LT" in sku:
        return "3 LITER"
    if "1LT" in sku or "1 LT" in sku or "1000ML" in sku or "1000 ML" in sku:
        return "1 LITER"
    return "OTHER"


# Menyiapkan data bulanan per SKU: Mar-Aug clean baseline solid, Sep forecast putus-putus
group_rows = []
for sku, sku_rows in forecast_detail_sep.groupby("SKU Edit"):
    group = pack_group_from_sku(sku)
    sep_value = float(sku_rows["Sep Forecast Base"].sum())
    for month in forecast_input_months:
        mon = month.strftime("%b")
        group_rows.append({
            "Pack Group": group,
            "SKU Edit": sku,
            "Month_Date": month,
            "Month Label": mon,
            "CF": float(sku_rows[f"{mon}_Clean"].sum()),
            "Point Type": "History",
            "Sep Forecast Base": sep_value,
        })
    group_rows.append({
        "Pack Group": group,
        "SKU Edit": sku,
        "Month_Date": FORECAST_MONTH,
        "Month Label": FORECAST_MONTH.strftime("%b"),
        "CF": sep_value,
        "Point Type": "Forecast",
        "Sep Forecast Base": sep_value,
    })

sku_group_monthly = pd.DataFrame(group_rows)
group_order = (
    sku_group_monthly[sku_group_monthly["Point Type"].eq("Forecast")]
    .groupby("Pack Group", as_index=False)
    .agg(sep_base=("CF", "sum"), sku_count=("SKU Edit", "nunique"))
    .query("sep_base > 0 and `Pack Group` != 'OTHER'")
    .sort_values("sep_base", ascending=False)
)

display(group_order)
display(sku_group_monthly.head(30))

palette = ["#005f73", "#2ca25f", "#f59e0b", "#6d5dfc", "#ef476f", "#118ab2"]

if plt is not None:
    for _, group_row in group_order.iterrows():
        group = group_row["Pack Group"]
        group_data = sku_group_monthly[sku_group_monthly["Pack Group"].eq(group)].copy()
        top_group_skus = (
            group_data[group_data["Point Type"].eq("Forecast")]
            .sort_values("CF", ascending=False)
            .head(5)["SKU Edit"]
            .tolist()
        )
        chart_data = group_data[group_data["SKU Edit"].isin(top_group_skus)]
        color_map = {sku: palette[i % len(palette)] for i, sku in enumerate(top_group_skus)}

        fig, ax = plt.subplots(figsize=(15, 4.8))
        for i, sku in enumerate(top_group_skus):
            sku_data = chart_data[chart_data["SKU Edit"].eq(sku)].sort_values("Month_Date")
            hist = sku_data[sku_data["Point Type"].eq("History")]
            fcst = sku_data[sku_data["Point Type"].eq("Forecast")]
            color = color_map[sku]

            ax.plot(hist["Month_Date"], hist["CF"], marker="o", linewidth=2.3, color=color, label=sku)
            if not hist.empty and not fcst.empty:
                ax.plot(
                    [hist["Month_Date"].max(), FORECAST_MONTH],
                    [float(hist.loc[hist["Month_Date"].idxmax(), "CF"]), float(fcst["CF"].iloc[0])],
                    linestyle="--",
                    marker="o",
                    linewidth=2.3,
                    color=color,
                )
                ax.annotate(
                    f"{float(fcst['CF'].iloc[0]):,.0f}",
                    xy=(FORECAST_MONTH, float(fcst["CF"].iloc[0])),
                    xytext=(8, 0),
                    textcoords="offset points",
                    fontsize=9,
                    fontweight="bold",
                    color=color,
                    va="center",
                )

        # Label history dibuat setelah semua garis digambar agar offset bisa diatur per bulan
        label_group_points_no_overlap(
            ax,
            chart_data[chart_data["Point Type"].eq("History")],
            color_map,
        )

        ax.set_title(f"{group} - {len(top_group_skus)} SKU", fontsize=14, fontweight="bold", loc="left")
        ax.set_ylabel("CF")
        ax.yaxis.set_major_formatter(FuncFormatter(fmt_k))
        ax.margins(y=0.18)
        ax.grid(True, axis="y", alpha=0.25)
        ax.grid(True, axis="x", alpha=0.10)
        ax.legend(loc="center left", bbox_to_anchor=(1.01, 0.5), frameon=False, fontsize=9)
        plt.tight_layout()
        plt.show()

## 22. Export Forecast CSV

File CSV akan disimpan di folder `forecast_output` pada folder yang sama dengan file dashboard di Google Drive.

In [ ]:
# ==========================================
# 23. 22. EXPORT FORECAST CSV
# ==========================================
# Menyimpan hasil forecast detail dan summary ke file CSV di Google Drive.

# Folder output dibuat di sebelah file dashboard agar mudah ditemukan di Google Drive
EXPORT_DIR = DRIVE_SALES_FILE.parent / "forecast_output"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# Label bulan forecast dipakai di nama file agar output tidak saling tertimpa antar periode
forecast_label = FORECAST_MONTH.strftime("%Y_%m")

# ==========================================
# 1. TABEL DATA ORDER HASIL CLEANING & ADJUSTMENT
# ==========================================
# Mengambil data order yang sudah memakai SKU Edit, CF Edit, dan status CF Adjustment untuk scope MODERN
order_clean_adjusted = forecast_source[[
    "Source Sheet", "Year", "Month No", "Date",
    "Channel Group", "Original Customer Name", "Customer Name", "Original Branch", "Branch", "Channel", "Cust Code",
    "Short Item Description Real", "Item Code Real",
    "Short Item Description Edit", "Item Code Edit",
    "Format", "Box Content", "Reference Box Content",
    "CF Real", "CF Edit", "CF Adjustment",
    "Group Key", "SKU Real", "SKU Edit", "Real != Edit",
]].copy()

# ==========================================
# 2. TABEL PIVOT CUSTOMER X SKU + PERHITUNGAN FORECAST
# ==========================================
# Membuat matrix actual Jan-Dec 2026 pada grain Customer x SKU
export_months = pd.date_range("2026-01-01", "2026-12-01", freq="MS")
export_month_labels = [m.strftime("%b_Actual") for m in export_months]

actual_2026_pivot = (
    monthly_customer_sku[monthly_customer_sku["Month"].dt.year.eq(2026)]
    .pivot_table(
        index=key_cols,
        columns="Month",
        values="CF",
        aggfunc="sum",
        fill_value=0,
    )
    .reset_index()
)

# Menambahkan bulan yang belum tersedia agar struktur file tetap Jan-Dec
for month in export_months:
    if month not in actual_2026_pivot.columns:
        actual_2026_pivot[month] = 0

# Rename kolom bulan menjadi Jan_Actual, Feb_Actual, dst.
actual_2026_pivot = actual_2026_pivot[
    key_cols + list(export_months)
].rename(columns={month: label for month, label in zip(export_months, export_month_labels)})

# Kolom kalkulasi forecast yang perlu diaudit di pivot
forecast_calc_cols = [
    "Mar_Clean", "Apr_Clean", "May_Clean", "Jun_Clean", "Jul_Clean", "Aug_Clean",
    "Aug_MTD", "Aug_RunRate", "Aug_Clean_RunRate",
    "Promo_Suspect_Months", "Promo_Suspect_Uplift",
    "Active Months", "Inactive 3M Rule", "Volatility", "Demand Class",
    "Median Recent Non-Promo", "Median Non-Promo", "Recent Non-Promo Avg",
    "Trimmed Mean Non-Promo", "Raw Forecast", "Guardrail Cap",
    "Current Month Guardrail",
    "Sep Forecast Base", "Sep Forecast Low", "Sep Forecast High",
]

# Menggabungkan matrix bulan dengan hasil perhitungan forecast
pivot_customer_sku_forecast = actual_2026_pivot.merge(
    forecast_detail_sep[key_cols + forecast_calc_cols],
    on=key_cols,
    how="left",
)

# ==========================================
# 3. TABEL FORECAST SEPTEMBER PER CUSTOMER X SKU
# ==========================================
# Mengambil kolom final yang dibutuhkan untuk upload/review forecast September
forecast_sep_customer_sku = forecast_detail_sep[[
    "Channel Group", "Branch", "Cust Code", "Customer Name",
    "Item Code Edit", "Short Item Description Edit", "SKU Edit",
    "Demand Class", "Inactive 3M Rule",
    "Sep Forecast Low", "Sep Forecast Base", "Sep Forecast High",
]].copy()
forecast_sep_customer_sku.insert(0, "Forecast Month", FORECAST_MONTH.strftime("%Y-%m"))

# Daftar 3 output forecast yang akan disimpan sebagai CSV
export_files = {
    f"01_order_clean_adjusted_{forecast_label}_modern.csv": order_clean_adjusted,
    f"02_pivot_customer_sku_forecast_{forecast_label}_modern.csv": pivot_customer_sku_forecast,
    f"03_forecast_sep_customer_sku_{forecast_label}_modern.csv": forecast_sep_customer_sku,
}

# Menulis semua dataframe ke CSV tanpa index pandas
exported_paths = []
for filename, dataframe in export_files.items():
    output_path = EXPORT_DIR / filename
    dataframe.to_csv(output_path, index=False, encoding="utf-8-sig")
    exported_paths.append(str(output_path))

# Menampilkan daftar file yang berhasil dibuat
print("Export folder:", EXPORT_DIR)
print("Exported CSV files:")
for path in exported_paths:
    print("-", path)